# Week 59, Day 1: Iterables and iterators

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w059d1.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
squares = map(lambda x: x * x, [1, 2, 3, 4])
print(3 * 3 in squares)
print(list(squares))
nums = [1, 2, 3]
it = iter(nums)
print(next(it), list(it), next(it, "done"))

In [ ]:
import io

class Squares:
    """Iterable through the sequence protocol alone."""
    def __init__(self, n):
        self.n = n
    def __getitem__(self, i):
        if not 0 <= i < self.n:
            raise IndexError(i)
        return i * i
    def __len__(self):
        return self.n

print(list(Squares(5)), list(reversed(Squares(5))), 9 in Squares(5))

f = io.StringIO("abcdefghij")
print(list(iter(lambda: f.read(4), "")))

In [ ]:
from collections.abc import Iterable, Iterator

for thing in ([1, 2], range(3), {"a": 1}.items(), zip("ab", "cd"), map(str, [1]), iter("xy"), 42):
    print(type(thing).__name__, isinstance(thing, Iterable), isinstance(thing, Iterator))

In [ ]:
nums = [1, 2, 2, 3, 2, 4]
for x in nums:
    if x == 2:
        nums.remove(x)
print(nums)                                   # one 2 survives

nums = [1, 2, 2, 3, 2, 4]
print([x for x in nums if x != 2])            # build a new list instead

stock = {"tea": 0, "milk": 3, "jam": 0}
for item in list(stock):                      # loop over a copy of the keys
    if stock[item] == 0:
        del stock[item]
print(stock)

In [ ]:
class Peekable:
    _EMPTY = object()

    def __init__(self, iterable):
        self._it = iter(iterable)
        self._held = self._EMPTY

    def __iter__(self):
        return self

    def __next__(self):
        if self._held is not self._EMPTY:
            value, self._held = self._held, self._EMPTY
            return value
        return next(self._it)

    def peek(self, default=None):
        if self._held is self._EMPTY:
            self._held = next(self._it, self._EMPTY)
        return default if self._held is self._EMPTY else self._held

p = Peekable("abc")
print(p.peek(), next(p), p.peek(), list(p), p.peek("end"))

> **Key idea.** for, list, sum, in and unpacking all call iter() and then next() until StopIteration. Iterables return a fresh iterator from __iter__ and can be looped over again; iterators return themselves and are used up after one pass. Classes can also be iterable through __getitem__ with IndexError, and iter(callable, sentinel) turns repeated calls into an iterator. Use collections.abc.Iterable and Iterator, or try iter(x), to tell them apart: range and dict views are iterables; map, filter, zip, enumerate, files and itertools are iterators. Never change a collection while looping over it: loop over a copy or build a new one. Iterators that wrap iterators stay lazy, so chains of them can handle endless or huge streams.

### Exercise: Reading in blocks

Write `blocks(stream, size)`: a list of the strings read from a file-like object in blocks of `size` characters, using `iter` with a callable and the sentinel `""`. Then `lines_until(stream, stop)`: the lines (without their newline) read with `stream.readline` until a line equal to `stop + "\n"` or the end, also with `iter`.

In [ ]:
import io

def blocks(stream, size):
    ___

def lines_until(stream, stop):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import io\ndef test_blocks():\n    \"iter with a sentinel\"\n    assert blocks(io.StringIO(\"abcdefghij\"), 4) == [\"abcd\", \"efgh\", \"ij\"] and blocks(io.StringIO(\"\"), 3) == []\n    assert lines_until(io.StringIO(\"one\\ntwo\\nEND\\nthree\\n\"), \"END\") == [\"one\", \"two\"]\n    assert lines_until(io.StringIO(\"a\\nb\\n\"), \"END\") == [\"a\", \"b\"]\n    assert \"iter(\" in source and \"while\" not in source")

### Exercise: A sequence without __iter__

Write a class `Times(n, upto)`: the multiplication table of n, `n × 1` to `n × upto`, iterable **only** through `__getitem__` (raising `IndexError` past the end) and `__len__`; no `__iter__`. Then add `__reversed__` returning an iterator that counts down, without building a list.

In [ ]:
class Times:
    def __init__(self, n, upto):
        ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("from collections.abc import Iterable\ndef test_table():\n    \"the sequence protocol\"\n    t = Times(3, 5)\n    assert list(t) == [3, 6, 9, 12, 15] and len(t) == 5 and 12 in t and 13 not in t\n    assert list(reversed(t)) == [15, 12, 9, 6, 3] and list(Times(7, 0)) == []\n    assert t[0] == 3 and t[4] == 15\n    try:\n        t[5]\n        assert False\n    except IndexError:\n        pass\n    assert not hasattr(Times, \"__iter__\") and not isinstance(t, Iterable)\n    r = reversed(Times(2, 1_000_000))\n    assert next(r) == 2_000_000 and not isinstance(r, list)")

### Exercise: Iterable or iterator?

Write `kind(obj)`: `"iterator"` if obj is an iterator, `"iterable"` if it can be looped over but isn't an iterator (including things iterable only through `__getitem__`), and `"neither"` otherwise. Use `collections.abc.Iterator`, and try `iter(obj)` for the rest.

In [ ]:
from collections.abc import Iterator

def kind(obj):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_kinds():\n    \"telling them apart\"\n    class Seq:\n        def __getitem__(self, i):\n            if i > 2:\n                raise IndexError\n            return i\n    assert [kind(x) for x in ([1], \"ab\", range(3), {}.items(), zip(), map(str, []), iter([]), 5, None, Seq())] == \\\n        [\"iterable\", \"iterable\", \"iterable\", \"iterable\", \"iterator\", \"iterator\", \"iterator\", \"neither\", \"neither\", \"iterable\"]\n    import io\n    assert kind(io.StringIO(\"x\")) == \"iterator\" and kind(enumerate([])) == \"iterator\" and kind(reversed([1])) == \"iterator\"")

### Exercise: Looking ahead

Write the class `Peekable(iterable)`: an iterator with `peek(default=...)` (the next item without consuming it; with no default, raise `StopIteration` at the end), `has_more()`, and `take_while(pred)`: a list of items from the front while `pred(item)` is True, leaving the first failing item **unconsumed**. Use a private sentinel object for "nothing held".

In [ ]:
class Peekable:
    def __init__(self, iterable):
        ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import itertools\ndef test_peek():\n    \"one item in hand\"\n    p = Peekable([1, 2, 3, 10, 4])\n    assert p.peek() == 1 and next(p) == 1 and p.has_more()\n    assert p.take_while(lambda x: x < 5) == [2, 3] and p.peek() == 10 and list(p) == [10, 4]\n    assert not p.has_more() and p.peek(\"end\") == \"end\"\n    try:\n        p.peek()\n        assert False\n    except StopIteration:\n        pass\n    q = Peekable([None, 0])\n    assert q.peek() is None and q.has_more() and next(q) is None and next(q) == 0 and not q.has_more()\n    endless = Peekable(itertools.count())\n    assert endless.take_while(lambda x: x < 3) == [0, 1, 2] and endless.peek() == 3")

### Exercise: In groups of n

Write an iterator class `Chunks(iterable, n)` whose items are **tuples** of n consecutive items from the iterable (the last may be shorter), lazily: it must work on an endless iterator. Raise `ValueError` if n < 1. Then `Window(iterable, n)`: tuples of every run of n neighbouring items (sliding by one), also lazily, using a `collections.deque` with `maxlen`.

In [ ]:
from collections import deque

class Chunks:
    def __init__(self, iterable, n):
        ___

class Window:
    def __init__(self, iterable, n):
        ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import itertools\ndef test_chunks():\n    \"lazy wrappers\"\n    assert list(Chunks(\"abcdefg\", 3)) == [(\"a\", \"b\", \"c\"), (\"d\", \"e\", \"f\"), (\"g\",)] and list(Chunks([], 2)) == []\n    assert list(Window([1, 2, 3, 4], 2)) == [(1, 2), (2, 3), (3, 4)] and list(Window([1, 2], 3)) == []\n    c = Chunks(itertools.count(), 2)\n    assert next(c) == (0, 1) and next(c) == (2, 3) and iter(c) is c\n    w = Window(itertools.count(), 3)\n    assert next(w) == (0, 1, 2) and next(w) == (1, 2, 3)\n    try:\n        Chunks([1], 0)\n        assert False\n    except ValueError:\n        pass")

### Bug hunt: Removing while looping

This removes the out-of-date entries while looping over the same list, so items shift under the loop and one is skipped: it prints `['new', 'old', 'new']`. Build a new list instead so it prints `['new', 'new']`.

In [ ]:
entries = ["old", "old", "new", "old", "new"]
for e in entries:
    if e == "old":
        entries.remove(e)
print(entries)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"a new list\"\n    expect_output(\"['new', 'new']\")")

### Bug hunt: An iterator, used twice

`filter` returns an iterator, so the first `len(list(...))` uses it up and the loop afterwards sees nothing: this prints `3` and then nothing at all. Keep the results in a list, so it prints `3` and then the three names.

In [ ]:
names = ["ada", "bo", "cy", "di"]
short = filter(lambda n: len(n) == 2, names)
print(len(list(short)))
for n in short:
    print(n)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"keep a list\"\n    expect_output(\"3\\nbo\\ncy\\ndi\")")

### Bug hunt: __iter__ must return an iterator

`__iter__` returns the list itself, which is iterable but not an **iterator**, so `for` raises `TypeError: iter() returned non-iterator of type 'list'`. Return `iter(self._items)` so it prints `a b c`.

In [ ]:
class Bag:
    def __init__(self, items):
        self._items = list(items)
    def __iter__(self):
        return self._items

try:
    print(*Bag("abc"))
except TypeError as e:
    print(type(e).__name__)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"an iterator\"\n    expect_output(\"a b c\")")

### Puzzle: An iterator that wraps another

Arrange the lines into an iterator class that skips every item equal to `skip`. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # list(zip("ab", range(5)))
q = ___   # r = range(3); list(r) + list(r)
r = ___   # z = zip("ab", "cd"); list(z) + list(z)
s = ___   # next(iter([]), "empty")
t = ___   # True or False: a dict's .items() is an iterator

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == [(\"a\", 0), (\"b\", 1)] and q == [0, 1, 2, 0, 1, 2] and r == [(\"a\", \"c\"), (\"b\", \"d\")], \"p, q and r\"\n    assert s == \"empty\" and t is False, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: merging streams lazily

Several sensors each produce readings in increasing time order, as iterators that may be very long or endless. Write two iterator classes, with no generators (that's tomorrow) and no lists of the inputs:

- `Interleave(*iterables)`: take one item from each in turn, skipping any that have run out, until all have.
- `MergeSorted(*iterables, key=None)`: all items in sorted order, assuming each input is already sorted, lazily, with a heap holding **one** item from each input at a time. Ties come out in the order of the inputs. With `key`, compare `key(item)`.

The checks feed endless iterators.

In [ ]:
import heapq

class Interleave:
    def __init__(self, *iterables):
        ___

class MergeSorted:
    def __init__(self, *iterables, key=None):
        ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import itertools, random\ndef test_streams():\n    \"lazy merging\"\n    assert list(Interleave(\"abc\", \"1\", \"xy\")) == [\"a\", \"1\", \"x\", \"b\", \"y\", \"c\"] and list(Interleave()) == []\n    assert list(MergeSorted([1, 4, 7], [2, 5], [3, 6, 9])) == [1, 2, 3, 4, 5, 6, 7, 9] and list(MergeSorted([], [])) == []\n    pairs = list(MergeSorted([(1, \"a\"), (3, \"a\")], [(1, \"b\"), (2, \"b\")], key=lambda p: p[0]))\n    assert pairs == [(1, \"a\"), (1, \"b\"), (2, \"b\"), (3, \"a\")]\n    evens, odds = itertools.count(0, 2), itertools.count(1, 2)\n    m = MergeSorted(evens, odds)\n    assert [next(m) for _ in range(6)] == [0, 1, 2, 3, 4, 5]\n    i = Interleave(itertools.count(), \"ab\")\n    assert [next(i) for _ in range(6)] == [0, \"a\", 1, \"b\", 2, 3]\n    rng = random.Random(1)\n    for _ in range(50):\n        lists = [sorted(rng.randint(0, 20) for _ in range(rng.randint(0, 6))) for _ in range(rng.randint(0, 4))]\n        assert list(MergeSorted(*lists)) == sorted(x for l in lists for x in l)\n    assert \"yield\" not in source")

## Warm-up

`map` returns an **iterator**: checking `in` consumed it up to the 9, so only 16 was left. A list is an **iterable**: each `iter(nums)` starts afresh. Mixing these up causes some of the most puzzling bugs in Python, so this week starts by making the difference automatic.

## The protocol, again

From [Week 36 Day 3](w036d3.html): `for x in things` calls `iter(things)`, which calls `things.__iter__()` and returns an **iterator**; then it calls `next(iterator)` (that is, `iterator.__next__()`) until `StopIteration` is raised. An **iterable** has `__iter__` returning a fresh iterator; an **iterator** has `__next__` and an `__iter__` that returns itself, which is why an iterator can be looped over only once.

Everything that consumes values uses the protocol: `for`, `list()`, `sum()`, `sorted()`, `in`, `*` unpacking, `zip`, comprehensions, `str.join`. So anything you make iterable works with all of them.

## Other ways to be iterable

`__iter__` isn't the only route:

- **The sequence protocol.** If a class has no `__iter__` but has `__getitem__` accepting 0, 1, 2, ... and raising `IndexError` at the end, `iter()` builds an iterator that calls it with increasing indexes. Old code relies on this; new code should define `__iter__`.
- **iter with a sentinel.** `iter(callable, sentinel)` calls the function with no arguments again and again, stopping when it returns the sentinel. It turns "read until empty" loops into iterators: `iter(lambda: f.read(1024), "")` yields blocks of a file until the end.
- **`next(iterator, default)`** returns the default instead of raising `StopIteration`.
- **`reversed(x)`** uses `x.__reversed__()` if there is one, otherwise the sequence protocol with `__len__` and `__getitem__`.

## Which is which?

The abstract base classes in `collections.abc` (Week 32) answer the question in code: `isinstance(x, Iterable)` is True if x has `__iter__`; `isinstance(x, Iterator)` if it has both `__iter__` and `__next__`. (An object that's iterable only through `__getitem__` isn't recognised as `Iterable`; the reliable test is to try `iter(x)` and catch `TypeError`.)

| Iterables (loop again and again) | Iterators (one pass only) |
|---|---|
| list, tuple, str, dict, set, range | `iter(...)` of any of them |
| dict views: `keys()`, `values()`, `items()` | `map`, `filter`, `zip`, `enumerate`, `reversed` |
| your classes with `__iter__` returning something new | open files, `csv.reader`, everything in `itertools` |

`range` is a famous surprise: it's an iterable, not an iterator, so `r = range(3)` can be looped over twice. `zip` is the opposite.

## Don't change it while you loop

An iterator over a list keeps an **index**; over a dict or set, it walks the hash table. Changing the collection underneath breaks the walk:

- Removing from a list while looping over it **skips** items (the index moves on while the items shift back), silently.
- Adding to or removing from a dict or set while looping raises `RuntimeError: dictionary changed size during iteration`.

The fixes: loop over a **copy** (`for x in list(items):`), build a **new** collection (a comprehension), or collect what to change and change it after the loop.

## Iterators that wrap iterators

The most useful iterators take **another** iterator and transform it, one item at a time, without ever building a list: `map`, `filter`, `zip` and `enumerate` all work like this. You can write your own: keep the inner iterator, and in `__next__` ask it for what you need. Because nothing runs until someone asks for the next item, chains of wrappers stay lazy: they can process an endless stream, or a file far bigger than memory.

A **peekable** iterator, which lets you look at the next item without consuming it, is a common example: it keeps one item in hand.

`_EMPTY = object()` is a **sentinel**: a unique object that can't be confused with any real item, even `None`.

## Practice

## Quiz

## Daily challenge

## Before you go

Find three loops in your own code from earlier weeks: is each looping over an iterable or an iterator, and could it be looping over the same iterator twice by mistake? Tomorrow: **generators**, where `yield` turns every class you wrote today into a short function.